# Troubleshooting, Monitoring & Optimization

## Reference domain

The bank's `silver_build` task in `fintech_nightly_ingest` (notebook 06) used to finish in 12 minutes. Last week it started taking 38 minutes. This notebook walks through the diagnostic flow — run history → task graph → Spark UI → physical plan — and the fix patterns Databricks expects you to apply.

## The diagnostic flow — where to look first

When a job starts misbehaving, follow this sequence. The exam asks variants of it directly.

```text
  ① Lakeflow Jobs — run history view
       Q: Is THIS run slow, or have all recent runs been slow?
       Output: a trend line — "started getting slow three days ago"
                                  │
                                  ▼
  ② Lakeflow Jobs — task graph view of one slow run
       Q: Which TASK is slow, and is it slow because an upstream is slow?
       Output: "silver_build is the slow task"
                                  │
                                  ▼
  ③ Spark UI for the slow task — Stages tab
       Q: Which STAGE is slow, and what does its task-level summary say?
       Output: "the join stage takes 30 minutes;
                most tasks finish in 30s but one task takes 25m"
                                  │
                                  ▼
  ④ Stage detail — task summary metrics
       Min/Median/Max shuffle read, spill, GC time
       Output: "Max shuffle read is 5 GB vs median 400 MB — that's skew"
                                  │
                                  ▼
  ⑤ Fix — match the pattern to the right remedy (the rest of this notebook)
```

The exam loves Step 4. "Most tasks finish quickly but one takes much longer; Max shuffle read is 10× the median." That sentence is **skew**, and the right answer is **AQE skew-join** (or salt + broadcast).

## Lakeflow Jobs run history — trend, not snapshot

The **Lakeflow Jobs run history view** lists every run of a job with its state, start time, duration, and a sparkline of recent durations. The single most important thing it shows you: **trend against historical baseline**.

Three patterns to recognise:

- **Gradual ramp** — duration creeping up over weeks. Usually data growth outpacing cluster capacity, or a slowly-growing fact table making a join progressively skewed.
- **Step change** — a single day where the curve jumps. Usually a code or config change, a new upstream source, or a runtime upgrade.
- **Spiky** — most runs at 10 min, occasional run at 90 min. Usually concurrency / resource contention, or an occasional bad data day.

The exam's framing: *"A job's median duration tripled overnight. Where do you look first?"* → **Lakeflow Jobs run history view**, then **task graph** of a slow run.

## Spark UI — what the tabs actually tell you

The Spark UI for a notebook task is your microscope. Three tabs the exam cares about.

**Jobs tab** — one row per action. The slow job is the slow row.

**Stages tab** — one row per stage of every job. Stages are the unit between shuffles. The slow stage is the slow row. Click into it.

**Stage detail page** — the gold. Two summaries to read:

- **Tasks table** — a row per task, sortable by duration / shuffle read / spill.
- **Task summary metrics** — min, 25th, median, 75th, max, plus mean — for duration, GC time, shuffle read, shuffle write, spill, and more.

**What each task-summary signal means:**

| Signal | What it tells you |
|---|---|
| Max task duration ≫ median | **Skew** — one partition is much bigger than the rest |
| Total shuffle write very large | **Shuffle-heavy** — joins / aggregations on big data |
| Spill to disk > 0 | **Memory pressure** — partitions don't fit in executor memory |
| GC time a large fraction of task time | **Memory pressure**, possibly OOM imminent |
| Many tiny tasks finishing in <100ms | **Partition over-fragmentation** — too many shuffle partitions |

**The Executors tab** is where you check per-executor memory and disk pressure, and look for failed / dead executors that point at OOM.

## Bottleneck 1 — Data skew

**Symptom in the UI:** stage duration is 25 minutes; 199 of 200 tasks finished in 30 seconds; one task is still running at 24 minutes. Task summary: median shuffle read 400 MB, max shuffle read 5 GB.

**Diagnosis:** the join or aggregation key is unbalanced. One value of `customer_id` (a corporate customer with 50 million transactions) has 100× the rows of the median customer. After the shuffle, all those rows land on one executor task.

**Three remedies, in the order the exam expects you to try them:**

1. **Enable AQE skew-join handling** — it's the *correct* answer for most exam questions. AQE detects oversized partitions at runtime and splits them into smaller sub-partitions, then joins each sub-partition independently. Zero code change.

    ```python
     spark.conf.set("spark.sql.adaptive.enabled", "true")
     spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
    ```

2. **Broadcast** the small side if it fits in driver memory — eliminates the shuffle entirely (notebook 04).

3. **Salt the join key** when AQE isn't enough. Add a random `0..N-1` integer to the key, join, then aggregate away the salt. Heavier surgery — keep it for when AQE has been tried and isn't sufficient.

    ```python
     from pyspark.sql.functions import lit, rand, floor
     # Add salt to the big side
     N = 8
     big = big_df.withColumn("salt", floor(rand() * N))
     # Explode the small side N times, once per salt value
     small_salted = (small_df
         .crossJoin(spark.range(N).withColumnRenamed("id", "salt")))
     joined = big.join(small_salted, on=["key", "salt"], how="inner")
    ```

**The exam answer 99% of the time is option 1** — enable AQE skew-join. The cited sample question from the exam guide names this as the right answer.

## Bottleneck 2 — Shuffling

**Symptom in the UI:** stage duration dominated by *Shuffle Write Time* / *Shuffle Read Time*; total shuffle write running into hundreds of GB; network is the bottleneck.

**Diagnosis:** joins and aggregations that require all rows for a key to land on the same executor cause a *shuffle* — a redistribution of data across the network.

**Remedies, in priority order:**

- **Broadcast** when one side fits in memory (notebook 04, notebook 08 section above).
- **Filter and project early** — the smallest shuffle is one that moves the fewest bytes. Push `WHERE` and `SELECT` upstream so the shuffle moves only what's needed.
- **Right-size `spark.sql.shuffle.partitions`** — for a 1 TB shuffle, 200 partitions means each partition is 5 GB (too big, spills). 4000 partitions means each is 250 MB (sweet spot).
- **Let AQE coalesce partitions** — `spark.sql.adaptive.coalescePartitions.enabled = true` merges tiny post-shuffle partitions into right-sized ones automatically.
- **Avoid unnecessary repartitioning** — calling `repartition(N)` before a join when the optimiser would have shuffled the right way anyway is a double shuffle.

## Bottleneck 3 — Disk spilling

**Symptom in the UI:** stage detail shows a non-zero **Spill (Memory)** or **Spill (Disk)** column. Executors are writing intermediate state to disk because it didn't fit in RAM.

**Why spilling hurts:** spilled data has to be re-read from disk, often de-serialised, and joined back in. That round trip is orders of magnitude slower than in-memory work.

**Remedies:**

- **Increase `spark.sql.shuffle.partitions`** — more, smaller partitions fit in memory.
- **Increase executor memory** — but watch the trade-off with fewer executors per node.
- **Filter / aggregate earlier** to reduce the per-partition payload.
- **Enable AQE** — coalesce partitions to the optimal size at runtime.

**The exam pattern:** *"a job is shuffling 200 GB across 200 partitions and tasks are spilling to disk."* The cleanest single-config answer is **bump `spark.sql.shuffle.partitions`** so each partition lands in the 100–200 MB range — about 1000–2000 partitions for 200 GB.

## AQE recap — what it does for you, automatically

**Adaptive Query Execution (AQE)** re-optimises the plan at runtime, after each shuffle, using *actual* partition statistics. Three things it does:

- **Dynamic partition coalescing** — merges undersized post-shuffle partitions. Fewer task-startup overheads, larger and more balanced units of work.
- **Dynamic join-strategy switching** — promotes a sort-merge join to a broadcast join when one side turns out smaller than estimated.
- **Skew-join handling** — detects oversized partitions and splits them so one slow task doesn't block the stage.

**AQE is on by default since Spark 3.2 / DBR 11.0.** On a modern Databricks workspace, you have it for free. The exam's correct answer to most performance questions is *check that AQE / its skew handling / its coalesce is enabled* — *not* manual tuning.

**Config keys to recognise:**

- `spark.sql.adaptive.enabled` — master switch.
- `spark.sql.adaptive.coalescePartitions.enabled`
- `spark.sql.adaptive.skewJoin.enabled`
- `spark.sql.adaptive.localShuffleReader.enabled`

## Liquid Clustering — operational, not just on-disk

Notebook 02 introduced Liquid Clustering as the modern replacement for partitioning + `ZORDER`. Here's the operational angle the exam tests.

**The pre-Liquid layout decision was one-shot.** Pick the partition column up front. Get it wrong, and re-partitioning is a full rewrite of TB of data. ZORDER helps with co-location *within* partitions, but partitioning itself stays fixed.

**Liquid Clustering changes the dynamic.** You declare the columns the table is clustered on; Delta organises data into clusters that can be incrementally re-balanced. You can change the keys later without rewriting the world.

```sql
 -- Initial creation
 CREATE TABLE silver.card_transactions (
    ...
 ) CLUSTER BY (customer_id, transaction_at);

 -- Three months later — query patterns shifted to merchant analysis
 ALTER TABLE silver.card_transactions
   CLUSTER BY (merchant_category, transaction_at);
```

**What it buys you operationally:**

- **No small-files-per-partition problem** — directories aren't the unit of organisation any more.
- **No skew when one partition is 100× larger** — clusters re-balance.
- **Evolvable** — change cluster keys as query patterns change.
- **Pairs with Predictive Optimization** — Databricks runs OPTIMIZE to maintain cluster balance automatically.

**The exam expects you to recognise this is the recommended default for new Delta tables.**

## Predictive Optimization — let Databricks run maintenance for you

Notebook 02 introduced **Predictive Optimization (PO)** as the way to skip writing maintenance jobs. The operational view:

**What it runs for you:** `OPTIMIZE` (compaction + cluster rebalancing) and `VACUUM` (storage reclamation), at the right cadence informed by your actual access patterns. Billed at a serverless rate.

**Enable at the catalog or schema level:**

```sql
 ALTER CATALOG fintech_prod ENABLE PREDICTIVE OPTIMIZATION;
```

**Two exam-relevant constraints:**

- **Unity Catalog managed tables only.** External tables don't get it. (One more reason to prefer managed.)
- **Specific cloud regions and DBR versions** — but the exam treats it as universally available.

**The exam question pattern:** *"how do you keep gold tables compact without writing a maintenance job?"* → enable Predictive Optimization.

## Cluster startup failures — the checklist

When a cluster fails to start, the workspace event log is the source of truth. The four most common failure modes:

**1. Cloud capacity / quota.** "InstanceLimitExceeded", "InsufficientCapacity". The cloud doesn't have the instance type available in that region/AZ, or your account's quota is exhausted. Fix: change instance type, change AZ, request quota increase, or switch to serverless.

**2. IAM / permissions.** "NotAuthorizedToAssumeRole", "403 from STS". The cluster's instance profile / service principal can't access the resources it needs. Fix: check the cluster role's trust policy and attached policies.

**3. Init scripts failing.** A `pip install` in an init script hits a network policy or a stale package. The cluster reports "INIT_SCRIPT_FAILURE". Fix: read the init script log under DBFS or the workspace's cluster logs, fix the script, restart. (Init scripts don't work on serverless — another reason serverless avoids this whole class of issues.)

**4. Library install failing.** A bad cluster library (PyPI, Maven) brings the cluster up but in `RESTARTING` or `ERROR`. Symptom: "library installation failed". Fix: identify the bad library in the cluster's *Libraries* tab, remove or pin a working version.

**The exam framing:** *"a job cluster fails to start — where do you look?"* → cluster's *Event Log* tab, then init script log.

## Library conflicts

Two PyPI packages requiring incompatible versions of the same dependency — the bane of every shared cluster. Symptoms:

- `ImportError` deep inside a third-party library.
- A library that worked yesterday breaks after the DBR upgrade.
- Two notebooks on the same cluster can each work, but together break.

**Resolution order:**

- **Use notebook-scoped Python (`%pip install pkg`)** instead of cluster libraries when only one notebook needs the dependency. Each notebook gets its own env.
- **Pin versions** in cluster libraries: `mypkg==2.4.1` not `mypkg`.
- **Use the DBR ML runtime** for ML packages — they're pre-resolved.
- **Use serverless compute** — Databricks resolves the runtime stack; you just `%pip install`.
- **For Java/Scala** — use a thin JAR with `--packages` or shading.

## Out-of-memory — executor vs. driver, and the fix patterns

**Executor OOM** — "`java.lang.OutOfMemoryError: Java heap space`" in a task log; tasks fail and the stage retries.

*Causes and fixes:*

- **Partition too big** → bump `spark.sql.shuffle.partitions` so each partition fits.
- **Skew packing too much into one task** → AQE skew-join.
- **`collect()` into a wide aggregation** → don't `collect`; write to a table.
- **UDF with high per-row memory** → batch with `pandas_udf` or rewrite as built-in functions.
- **Caching too much** → `unpersist` when done, or use disk-only caching.
- *Last resort:* bump `spark.executor.memory`.

**Driver OOM** — "`OutOfMemoryError`" reported from the driver; the notebook hangs or the cluster restarts.

*Causes and fixes:*

- **`collect()` / `toPandas()` on big DataFrames** → never do this on more than a few MB.
- **Massive Spark UI / event log** → reduce job complexity, or bump `spark.driver.memory`.
- **Broadcasting too-big tables** → cap `spark.sql.autoBroadcastJoinThreshold`.
- *Last resort:* bump `spark.driver.memory`.

**The exam pattern:** *"executor task fails with OOM after the shuffle."* → increase shuffle partitions; AQE skew-join; broadcast small side. Bumping memory is the last lever, not the first.

## Hands-on — diagnosing skew on the bank's silver build

We construct a deliberately skewed dataset, run a join, observe the slow task and Max ≫ Median pattern, then turn on AQE skew-join and re-measure. Run on a Databricks cluster (or `local[*]`) — DBR 14.x+ for AQE skew-join.

In [ ]:
from pyspark.sql.functions import col, lit, when

# Big side — 5 million transactions; 80% all belong to one 'corporate' customer (C_BIG).
big = (spark.range(5_000_000)
    .withColumn("customer_id", when(col("id") % 5 == 0, lit("C_BIG")).otherwise(col("id").cast("string")))
    .withColumn("amount", col("id") % 1000))

# Small side — a per-customer attribute. C_BIG is just one row.
small = (spark.range(1_000_000)
    .withColumn("customer_id", col("id").cast("string"))
    .withColumn("segment", lit("retail")))
small = small.union(spark.createDataFrame([("C_BIG", "corporate")], ["customer_id", "segment"]))

print("big rows  :", big.count())
print("small rows:", small.count())

In [ ]:
# Turn AQE off temporarily so we see the un-handled skew pattern.
import time
spark.conf.set("spark.sql.adaptive.enabled", "false")

t0 = time.time()
n = (big.join(small, on="customer_id")
        .groupBy("segment").sum("amount")
        .collect())
print(f"AQE off  — wall: {time.time() - t0:.1f}s")
print(n)
# In the Spark UI for this run, the join stage will show:
#   • 199 fast tasks, 1 very slow task
#   • Max shuffle read ≫ Median shuffle read
# That's the textbook skew pattern.

In [ ]:
# Turn AQE + skew-join on — the recommended fix.
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

t0 = time.time()
n = (big.join(small, on="customer_id")
        .groupBy("segment").sum("amount")
        .collect())
print(f"AQE on   — wall: {time.time() - t0:.1f}s")
print(n)

In [ ]:
# Even better when the small side really does fit — broadcast eliminates the shuffle entirely.
from pyspark.sql.functions import broadcast

t0 = time.time()
n = (big.join(broadcast(small), on="customer_id")
        .groupBy("segment").sum("amount")
        .collect())
print(f"Broadcast— wall: {time.time() - t0:.1f}s")
print(n)

In [ ]:
# Read the physical plan — AQE markers, BroadcastExchange, and skew labels are visible.
(big.join(small, on="customer_id")
    .groupBy("segment").sum("amount")
    .explain(mode="formatted"))

In [ ]:
# The full config-key recap — print the current values so you can compare.
for k in (
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.adaptive.skewJoin.enabled",
    "spark.sql.adaptive.localShuffleReader.enabled",
    "spark.sql.shuffle.partitions",
    "spark.sql.autoBroadcastJoinThreshold",
):
    print(f"{k:60s} = {spark.conf.get(k)}")

## Section 6 self-check

**1.** A job runs in 30 seconds for 199 tasks, but one task takes 10 minutes. Stage task summary: Min/Median shuffle read ≈ 400 MB; Max shuffle read > 5 GB. Which fix is recommended *first*?

- A. Increase cluster size to add more executors
- B. Confirm AQE with skew-join handling is enabled so the oversized partition splits at runtime
- C. Lower `spark.sql.shuffle.partitions` to coalesce work into fewer tasks
- D. Manually salt the join key

**2.** A job shuffling 200 GB across 200 partitions is spilling to disk. The single config change most likely to help?

- A. Lower `spark.sql.shuffle.partitions` to 8
- B. Raise `spark.sql.shuffle.partitions` to ~1000–2000 so each partition is ~100–200 MB
- C. Disable AQE
- D. Set `spark.sql.autoBroadcastJoinThreshold` to -1

**3.** Which feature keeps Unity Catalog managed Delta tables compacted and pruned automatically?

- A. Liquid Clustering
- B. Predictive Optimization
- C. Photon
- D. AQE

**4.** A job cluster fails to start with `InstanceLimitExceeded`. Which option is the right fix?

- A. Increase `spark.driver.memory`
- B. Switch instance type, switch AZ, request a quota increase, or move to serverless
- C. Disable AQE
- D. Use a high-concurrency cluster

**5.** Two notebooks share an all-purpose cluster; one breaks after the other runs `%pip install` of an incompatible library. How do you keep both working without coordinating?

- A. Use cluster libraries with unpinned versions
- B. Use notebook-scoped `%pip` so each notebook has its own Python environment, or use serverless
- C. Restart the cluster between notebook runs
- D. Use a single-node cluster

<details><summary>Show answers</summary>

1. **B** — AQE skew-join is the recommended first move (matches the official sample answer).
2. **B** — right-sized shuffle partitions reduce spill.
3. **B** — Predictive Optimization runs `OPTIMIZE` and `VACUUM`.
4. **B** — `InstanceLimitExceeded` is a cloud quota / capacity issue.
5. **B** — notebook-scoped pip isolates Python envs per notebook; serverless avoids the problem entirely.

</details>